# Qianfan-OCR 信息抽取能力Cookbook

本 Notebook展示若干常见场景下的信息抽取实践，覆盖发票、合同等典型文档。除以下示例外，模型同样可用于更多结构化/半结构化文档的字段提取与理解任务：
- 长列表发票明细解析
- 多图发票批量识别
- 多页合同关键信息提取

## 环境准备

In [27]:
!pip3 install openai -i https://pypi.tuna.tsinghua.edu.cn/simple/

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple/


In [ ]:
import base64
import json
from pathlib import Path
from openai import OpenAI
from IPython.display import display, Image, HTML

# API 配置
client = OpenAI(
    base_url="http://10.231.136.51:8080/v1",
    api_key="EMPTY"
)
MODEL = "v930k-v35-0309-6"

def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def get_media_type(path):
    suffix = Path(path).suffix.lower()
    return "image/png" if suffix == ".png" else "image/jpeg"

def ocr_extract(images, prompt):
    content = []
    for img in images:
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:{get_media_type(img)};base64,{encode_image(img)}"}
        })
    content.append({"type": "text", "text": prompt})
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": content}],
        max_tokens=4096
    )
    return response.choices[0].message.content

def show_images(images, titles=None, width=300):
    """横向展示图片"""
    html = '<div style="display:flex; gap:12px; flex-wrap:wrap; align-items:flex-start;">'
    for i, img_path in enumerate(images):
        title = titles[i] if titles else ""
        b64 = encode_image(img_path)
        media_type = get_media_type(img_path)
        html += f'''<div style="text-align:center;">
            <img src="data:{media_type};base64,{b64}" style="max-width:{width}px; border:1px solid #ccc;"/>
            <div style="color:#666; font-size:12px; margin-top:4px;">{title}</div>
        </div>'''
    html += '</div>'
    display(HTML(html))

def show_json(result):
    """简洁展示JSON结果"""
    try:
        parsed = json.loads(result)
        formatted = json.dumps(parsed, ensure_ascii=False, indent=2)
    except:
        formatted = result
    display(HTML(f'<pre style="background:#f5f5f5; padding:12px; border:1px solid #ddd; border-radius:4px; font-size:13px; color:#333; overflow-x:auto;">{formatted}</pre>'))

def show_invoice_table(result):
    """将发票明细渲染为HTML表格"""
    try:
        data = json.loads(result)
    except:
        print(result)
        return
    
    html = '<div style="font-size:13px; color:#333;">'
    
    # 基本信息表格
    basic_fields = ['开票日期', '发票种类', '发票代码', '发票号码']
    html += '<table style="border-collapse:collapse; margin:8px 0; width:100%;"><tr>'
    for field in basic_fields:
        val = data.get('基本信息', data).get(field, data.get(field, ''))
        if val:
            html += f'<td style="padding:6px 12px; border:1px solid #ddd; background:#f9f9f9;"><b>{field}</b>: {val}</td>'
    html += '</tr></table>'
    
    # 金额信息
    amount_info = data.get('金额信息', data)
    amount_fields = [('合计金额', '合计金额'), ('合计税额', '合计税额'), ('价税合计(小写)', '价税合计(小写)'), ('价税合计(大写)', '价税合计(大写)')]
    html += '<table style="border-collapse:collapse; margin:8px 0;"><tr>'
    for label, key in amount_fields:
        val = amount_info.get(key, data.get(key, ''))
        if val:
            html += f'<td style="padding:6px 12px; border:1px solid #ddd; background:#f9f9f9;"><b>{label}</b>: {val}</td>'
    html += '</tr></table>'
    
    # 明细表格
    items = data.get('明细信息', [])
    if items:
        html += '<table style="border-collapse:collapse; width:100%; margin:8px 0;">'
        html += '<tr style="background:#f0f0f0;">'
        headers = ['货物名称', '规格型号', '单位', '数量', '单价', '金额', '税率', '税额']
        for h in headers:
            html += f'<th style="padding:8px; border:1px solid #ddd; text-align:left; font-weight:500;">{h}</th>'
        html += '</tr>'
        for i, item in enumerate(items):
            bg = '#fff' if i % 2 == 0 else '#f9f9f9'
            html += f'<tr style="background:{bg};">'
            for h in headers:
                val = item.get(h, '')
                html += f'<td style="padding:6px 8px; border:1px solid #ddd;">{val}</td>'
            html += '</tr>'
        html += '</table>'
    
    html += '</div>'
    display(HTML(html))

print("环境准备完成")

---
## 场景一：发票长列表详细解析

**任务**：从包含多行货物明细的增值税发票中，提取发票基本信息、购销双方信息、金额汇总及逐行明细，结构化输出为 JSON 格式。

In [ ]:
long_invoice = ["../images/invoice_long_list/invoice_detail.png"]
show_images(long_invoice, titles=["长列表发票"], width=700)

In [30]:
prompt_1 = """请从图片中提取以下信息：

基本信息：发票代码、发票号码、开票日期、发票种类、发票抬头全称、校验码
购方信息：购方名称、购方纳税人识别号、购方地址电话、购方开户行账号
销售方信息：销售方名称、销售方纳税人识别号、销售方地址电话、销售方开户行账号
金额信息：合计金额、合计税额、价税合计(小写)、价税合计(大写)、小计金额、小计税额
明细信息：货物名称、规格型号、单位、数量、单价、金额、税率、税额
其他信息：收款人、复核、开票人、密码区、备注、二维码、代开、机器编号、联次、总页码、当前页码

注意：
1. 仅提取清晰可见的内容，模糊或缺失的部分请忽略
2. 保持原始格式，使用标准JSON输出"""

result_1 = ocr_extract(long_invoice, prompt_1)
show_invoice_table(result_1)

---
## 场景二：多张发票批量识别

**任务**：同时输入多张发票图片，批量识别并提取每张发票的"价税合计"金额，按发票顺序汇总输出。

In [ ]:
invoice_images = ["../images/multi_invoices/invoice_1.jpg", "../images/multi_invoices/invoice_2.jpg", "../images/multi_invoices/invoice_3.jpg"]
show_images(invoice_images, titles=["发票1", "发票2", "发票3"], width=320)

In [33]:
prompt_2 = """你是一个专业的发票信息提取AI助手，请从每张图像中识别并提取"价税合计"信息（即总金额，包括税费的小写形式，例如"1000.00"）。

输出格式为严格的JSON对象：
- 以发票序号作为键（从"发票1"开始递增）
- 值是提取到的价税合计信息（如果无法识别，则用"无法提取"表示）

仅输出JSON，不要添加额外文本。"""

result_2 = ocr_extract(invoice_images, prompt_2)
show_json(result_2)

---
## 场景三：多页合同信息抽取

**任务**：输入一份多页扫描合同（共6页），从中定位并提取结算账户相关信息，包括账户名称、开户银行、银行账号等关键字段。

In [ ]:
contract_images = [f"../images/multi_page_contract/contract_page_{i+1}.jpg" for i in range(6)]
show_images(contract_images, titles=[f"第{i+1}页" for i in range(6)], width=260)

In [35]:
prompt_3 = """你是一位合同信息抽取专家。请从输入的多页合同图片中，定位并抽取结算账户相关信息，并以 JSON 格式输出。

要求：
- 仅包含图片中明确出现的具体信息
- 如果某项信息未出现，则不输出该项
- 严禁进行任何推断、补全或添加未出现的内容"""

result_3 = ocr_extract(contract_images, prompt_3)
show_json(result_3)